In [ ]:


from __future__ import annotations

import argparse
from pathlib import Path
import sys

import cv2


def make_tracker():
    """Create a tracker across legacy and current OpenCV Python APIs."""
    candidates = ("TrackerCSRT_create", "TrackerKCF_create", "TrackerMOSSE_create")

    for name in candidates:
        fn = getattr(cv2, name, None)
        if fn:
            return fn()

    legacy = getattr(cv2, "legacy", None)
    if legacy:
        for name in candidates:
            fn = getattr(legacy, name, None)
            if fn:
                return fn()

    raise RuntimeError(
        "No CSRT/KCF/MOSSE tracker found. Install opencv-contrib-python."
    )


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("video", type=Path, help="Input match video")
    parser.add_argument("--output", type=Path, help="Annotated output video (optional)")
    parser.add_argument(
        "--players",
        action="store_true",
        help="Count people on each image half with YOLO",
    )
    parser.add_argument(
        "--model",
        default="yolov8n.pt",
        help="Ultralytics model/weights (default: yolov8n.pt)",
    )
    parser.add_argument(
        "--confidence",
        type=float,
        default=0.30,
        help="Player detection confidence (0..1)",
    )
    parser.add_argument(
        "--detect-every",
        type=int,
        default=3,
        help="Run player detector every N frames",
    )
    args = parser.parse_args()

    if not args.video.is_file():
        parser.error(f"Video not found: {args.video}")
    if not 0 <= args.confidence <= 1:
        parser.error("--confidence must be between 0 and 1")
    if args.detect_every < 1:
        parser.error("--detect-every must be at least 1")

    cap = cv2.VideoCapture(str(args.video))
    if not cap.isOpened():
        print(f"Could not open video: {args.video}", file=sys.stderr)
        return 2

    ok, first = cap.read()
    if not ok:
        print("Video contains no readable frames.", file=sys.stderr)
        cap.release()
        return 2

    window_name = "Select volleyball (drag box, Enter/Space to confirm)"
    roi = cv2.selectROI(window_name, first, showCrosshair=True, fromCenter=False)
    cv2.destroyWindow(window_name)

    x, y, w, h = map(int, roi)
    if w < 3 or h < 3:
        print("No valid ball selection made.", file=sys.stderr)
        cap.release()
        return 2

    tracker = make_tracker()
    tracker.init(first, (x, y, w, h))
    trail: list[tuple[int, int]] = []

    model = None
    if args.players:
        try:
            from ultralytics import YOLO

            model = YOLO(args.model)
        except Exception as exc:
            cap.release()
            print(f"Could not load YOLO model '{args.model}': {exc}", file=sys.stderr)
            return 2

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = None

    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        writer = cv2.VideoWriter(
            str(args.output),
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (width, height),
        )
        if not writer.isOpened():
            cap.release()
            print(f"Could not create output video: {args.output}", file=sys.stderr)
            return 2

    frame_index = 0
    counts = (0, 0)

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_index += 1
        tracked, box = tracker.update(frame)

        if tracked:
            bx, by, bw, bh = map(int, box)
            center = (bx + bw // 2, by + bh // 2)
            trail.append(center)
            trail = trail[-100:]

            cv2.rectangle(frame, (bx, by), (bx + bw, by + bh), (0, 255, 0), 2)
            cv2.circle(frame, center, 4, (0, 0, 255), -1)
            cv2.putText(
                frame,
                "Ball",
                (bx, max(20, by - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 255, 0),
                2,
            )
        else:
            cv2.putText(
                frame,
                "Ball lost - press q to stop",
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2,
            )

        if len(trail) > 1:
            import numpy as np

            cv2.polylines(
                frame,
                [np.array(trail, dtype="int32")],
                False,
                (255, 180, 0),
                2,
            )

        if model and frame_index % args.detect_every == 0:
            try:
                result = model.predict(
                    frame,
                    classes=[0],
                    conf=args.confidence,
                    verbose=False,
                )[0]
                boxes = (
                    result.boxes.xyxy.cpu().numpy()
                    if result.boxes is not None
                    else []
                )
                counts = (
                    sum(1 for b in boxes if (b[0] + b[2]) / 2 < width / 2),
                    sum(1 for b in boxes if (b[0] + b[2]) / 2 >= width / 2),
                )
            except Exception as exc:
                print(
                    f"Player detection failed at frame {frame_index}: {exc}",
                    file=sys.stderr,
                )
                model = None

        if model:
            cv2.line(
                frame,
                (width // 2, 0),
                (width // 2, height),
                (180, 180, 180),
                1,
            )
            cv2.putText(
                frame,
                f"Players: left {counts[0]} | right {counts[1]}",
                (15, height - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
            )

        if writer:
            writer.write(frame)

        cv2.imshow("Volleyball tracker (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    if writer:
        writer.release()
    cv2.destroyAllWindows()

    if args.output:
        print(f"Saved annotated video to {args.output}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())
```